# ChemBreak Adaptive Jailbreak v1

GitHub to Colab Enterprise runner for the ChemBreak repository.

Repository: `https://github.com/Jollychuks/ChemBreak`

Project folder: `ChemBreak_Adaptive_Jailbreak_v1`

Run TEST first. Use PILOT to freeze the attack budget, judge thresholds, route-switch limit, model revisions, and prompts. Run PRODUCTION only after those settings are frozen.


## 1. Clone or refresh the ChemBreak GitHub repository

This cell only changes the ephemeral Colab Enterprise runtime. It never pushes to GitHub.


In [ ]:
from pathlib import Path
import os, subprocess

REPO_URL = "https://github.com/Jollychuks/ChemBreak.git"
BRANCH = "main"
PROJECT_SUBDIR = "ChemBreak_Adaptive_Jailbreak_v1"

WORK_ROOT = Path("/content") if Path("/content").exists() else (Path.home() / "chembreak_colab")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR = WORK_ROOT / "ChemBreak"

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

PROJECT_DIR = REPO_DIR / PROJECT_SUBDIR
assert PROJECT_DIR.exists(), f"Project folder not found: {PROJECT_DIR}. Upload {PROJECT_SUBDIR} to the GitHub repo first."
os.chdir(PROJECT_DIR)
print("Repository:", REPO_DIR)
print("Project directory:", PROJECT_DIR)
print("Git commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


## 2. Install dependencies

The package does not request a new PyTorch installation. It uses the CUDA-enabled PyTorch already present in the Google Cloud runtime.


In [ ]:
!python -m pip install -q -r requirements.txt
!python -m pip install -q -e .

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))


## 3. Configure this run

`RUN_ID` must stay the same when resuming an interrupted run. Change it only when you intentionally start a fresh experiment namespace.


In [ ]:
GCP_PROJECT = "rs-foundsecft-mghasemi"
GCS_BUCKET = "rs-foundsecft-mghasemi-default-1"

RUN_MODE = "test"        # test | pilot | production
RUN_ID = "test_001"      # keep stable to resume this run

# The notebook searches this private GCS prefix for the completed V15 final task bank.
TASK_BANK_PREFIX = "ChemBreak_V15/outputs/production/"

# Durable checkpoints and restricted transcripts go here.
GCS_BASE_URI = f"gs://{GCS_BUCKET}/ChemBreak_Adaptive_Jailbreak_v1"
GCS_OUTPUT_URI = f"{GCS_BASE_URI}/{RUN_MODE}/{RUN_ID}"

print("Project:", GCP_PROJECT)
print("Mode:", RUN_MODE)
print("Run ID:", RUN_ID)
print("Output:", GCS_OUTPUT_URI)


## 4. Locate the final ChemBreak task bank in Google Cloud Storage


In [ ]:
from google.cloud import storage

client = storage.Client(project=GCP_PROJECT)
blobs = list(client.list_blobs(GCS_BUCKET, prefix=TASK_BANK_PREFIX))
matches = [b for b in blobs if b.name.lower().endswith(".csv") and "final_task_bank" in b.name.lower()]
matches.sort(key=lambda b: b.updated or 0, reverse=True)

if not matches:
    raise FileNotFoundError(
        f"No final_task_bank CSV found under gs://{GCS_BUCKET}/{TASK_BANK_PREFIX}. "
        "Change TASK_BANK_PREFIX or set TASK_BANK_URI manually."
    )

print("Matching final task banks:")
for i, b in enumerate(matches, 1):
    print(f"{i}. gs://{GCS_BUCKET}/{b.name} | updated={b.updated}")

TASK_BANK_URI = f"gs://{GCS_BUCKET}/{matches[0].name}"
print("\nSelected:", TASK_BANK_URI)


## 5. Create a local runtime configuration

`configs/runtime.yaml` is ignored by Git, so runtime paths do not get committed to the public repository.


In [ ]:
import subprocess
subprocess.run([
    "python", "scripts/create_runtime_config.py",
    "--template", "configs/gcp.yaml",
    "--output", "configs/runtime.yaml",
    "--project", GCP_PROJECT,
    "--run-mode", RUN_MODE,
    "--run-id", RUN_ID,
    "--task-bank-uri", TASK_BANK_URI,
    "--gcs-output-uri", GCS_OUTPUT_URI,
], check=True)

CONFIG = "configs/runtime.yaml"
print(Path(CONFIG).read_text()[:4000])


## 6. Preflight

Checks GPU access, task-bank schema, Hugging Face repositories and revisions, and benign Vertex attacker/judge calls. It does not send benchmark tasks to ChemDFM, ChemLLM, or LlaSMol.


In [ ]:
!python scripts/preflight.py --config $CONFIG


## 7. Prepare attack assets

For every selected task this creates the repeated single-turn attempts, fixed multi-turn sequence, and four-route adaptive attack graph once. These assets are reused across all three target models.


In [ ]:
!python scripts/run.py prepare --config $CONFIG


## 8. Run ChemDFM

This runs C0, C1, C2 and C3 for ChemDFM. The model is loaded once and reused across the whole target block.


In [ ]:
!python scripts/run.py execute --config $CONFIG --target chemdfm


## 9. Run ChemLLM


In [ ]:
!python scripts/run.py execute --config $CONFIG --target chemllm


## 10. Run LlaSMol


In [ ]:
!python scripts/run.py execute --config $CONFIG --target llasmol


## 11. Rebuild aggregate metrics

Execution already updates metrics after each target block. This cell is safe to rerun after any resumed execution.


In [ ]:
!python scripts/run.py metrics --config $CONFIG


## 12. Resume behavior

If the runtime stops, reconnect to Colab Enterprise, rerun Sections 1 through 6 with the same `RUN_MODE` and `RUN_ID`, then rerun prepare and the target cells. Completed task-target-condition units are skipped, and GCS checkpoints are downloaded before execution.
